In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/crop_production.csv")
df.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [2]:
print(df.shape)
df.info()

(153177, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153177 entries, 0 to 153176
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   State_Name     153177 non-null  object 
 1   District_Name  153177 non-null  object 
 2   Crop_Year      153177 non-null  int64  
 3   Season         153177 non-null  object 
 4   Crop           153176 non-null  object 
 5   Area           153176 non-null  float64
 6   Production     150499 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 8.2+ MB


In [3]:
print(df.isnull().sum())       # missing values per column
print(df.duplicated().sum())   # number of duplicate rows
df.describe()                  # min, max, mean of number columns

State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                1
Area                1
Production       2678
dtype: int64
0


,Crop_Year,Area,Production
count,153177.000000,153176.000000,1.504990e+05
mean,2005.778557,10735.763770,8.149586e+05
std,4.917471,34941.037138,2.071903e+07
min,1997.000000,0.040000,0.000000e+00
25%,2002.000000,96.000000,1.000000e+02
50%,2006.000000,600.000000,7.596100e+02
75%,2010.000000,4260.000000,6.744800e+03
max,2015.000000,877029.000000,1.125000e+09


In [4]:
df.columns = df.columns.str.strip().str.lower()   #Fix column names and text. Extra spaces and inconsistent capitals cause “Kharif” and “Kharif “ to count as two different values.
text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].str.strip().str.title()

In [6]:
# Remove duplicates.
before = len(df)
df = df.drop_duplicates()
print("Removed:", before - len(df))

Removed: 0


In [7]:
# Handle missing values.
print(df.isnull().sum())
# Option A: drop rows where the key number is missing
df = df.dropna(subset=["production"])
# Option B: fill a number column with its median
# df["area"] = df["area"].fillna(df["area"].median())

state_name          0
district_name       0
crop_year           0
season              0
crop                1
area                1
production       2678
dtype: int64


In [8]:
# Fix data types
df["area"] = pd.to_numeric(df["area"], errors="coerce")
df["production"] = pd.to_numeric(df["production"], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 150499 entries, 0 to 153175
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   state_name     150499 non-null  object 
 1   district_name  150499 non-null  object 
 2   crop_year      150499 non-null  int64  
 3   season         150499 non-null  object 
 4   crop           150499 non-null  object 
 5   area           150499 non-null  float64
 6   production     150499 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 9.2+ MB


In [9]:
# Find outliers with the IQR method. Outliers are extreme values that are far from the rest.
Q1 = df["production"].quantile(0.25)
Q3 = df["production"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["production"] < lower) | (df["production"] > upper)]
print("Outliers found:", len(outliers))

Outliers found: 25630


In [10]:
# Normalise and add a yield column.
df["yield"] = df["production"] / df["area"]
df["production_norm"] = (df["production"] - df["production"].min()) / (df["production"].max() - df["production"].min())

In [11]:
df.to_csv("cleaned_crop_production.csv", index=False)